In [6]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-study\chapter04
프로젝트 루트: C:\dev\llm-data-analysis-study
데이터 폴더: C:\dev\llm-data-analysis-study\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-study\reports


In [7]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')

데이터 불러오기 완료


In [8]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


**1. 분석 질문: 어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?**



In [9]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


In [10]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-14
8,9,송지민,M,69,서울,2025-12-29
54,55,윤영철,M,69,광주,2023-10-10
78,79,이서준,M,69,부산,2023-09-23
136,137,김선영,M,68,울산,2025-02-07
45,46,김서현,M,67,대전,2026-07-23
132,133,김성진,M,67,성남,2026-08-08
123,124,김시우,M,67,대구,2024-12-04
11,12,김정남,F,66,대전,2024-12-10
57,58,김성훈,F,66,성남,2025-08-24


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [11]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


**1-1. 결과 관찰**
분석 질문: 어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?
- 사용할 데이터와 주요 컬럼
order_items: order_id, product_id, quantity, unit_price
orders: order_id, order_status
products: product_id, category
- 포함할 주문 상태: order_status가 completed인 주문만 포함.
- 금액 계산식: line_total = quantity × unit_price로 계산.
- 최종 집계 기준: category별 line_total 합계

**1-2. 나의 해석과 판단**
금액이 제시되어 있는 order_items 데이터를 기준 테이블로 잡았다. 주문 상태는 orders 데이터에서 order_status 컬럼을, 카테고리는 products 데이터에 category 컬럼을 사용하기로 판단하였다. 단가는 주문 당시의 단가가 기록된 order_items 데이터의 unit_price 컬럼을 사용했다. 날짜나 고객 정보는 이번 분석 질문에 직접 필요해보이지 않아 제외하였다. 

**1-3. 업무·분석적 의미**
분석 질문, 포함 범위, 계산식, 집계 단위를 먼저 고정해두면 이후 결과가 어떤 기준의 숫자인지 설명하기 용이하다. 

**1-4. 한계**
금액은 정의한 대로 quantity x unit_price 기준이기 때문에 이 계산식에 포함되지 않은 요소들이 존재한다. unit_price가 어떤 가격 종류인지(정가? 주문 시점 가격?) 확인하지 못했다. 

**2. 실제 데이터 확인 및 병합**

In [12]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

**2-1. 결과 관찰**
- order_status 실제 값: completed 184건, cancelled 64건, refunded 52건, 결측 없음
- 병합 key: order_id
- orders 데이터의 order_id 중복 0건
- 병합 전 764행, 병합 후 764행이며 both 764행, left_only(미매칭) 0건.

**2-2. 나의 해석과 판단**
- orders 데이터의 key가 고유하기 때문에 many-to-one 병합이 가능하다고 판단하였다. 행 수가 그대로이며 left_only가 0건이므로, 병합으로 행이 늘거나 줄지 않았고 모든 order_items 데이터가 orders 데이터와 연결되었다. (미매칭 데이터 0건) 상태 값이 세 종류(completed, cancelled, refunded)뿐이고, 공백이나 대소문자 변형이 없기 때문에 =='completed' 정확 일치 필터를 써도 누락이 없다고 판단하였다. 

**2-3. 업무·분석적 의미**
- 병합 단계에서 행이 늘거나 줄어드는 현상과 미매칭이 없다는 것을 확인했기 때문에, 이후 금액 합계가 병합 오류 때문에 부풀거나 누락될 가능성을 낮출 수 있다.

**2-4. 한계**
- order_items 데이터를 기준으로 orders 데이터를 붙였기에, orders 데이터는 있는데 order_items 데이터 상세 행이 하나도 없는 주문이 있을 수도 있다. 즉, 상품 내역이 비어있는 주문이 있는지 확인하지 못했다. 금액에는 영향이 없지만, orders 기준으로 주문 건수를 셀 때는 상품 내역이 없는 주문이 건수에 포함될 수 있다. 

**3. completed 주문만 분석**

In [13]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


In [14]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

**3-1. 결과 관찰**
- order_status == 'completed'로 필터링한 결과 주문 상세 행은 전체 764행 중 474행이다. 
- 월별 order_count를 합하면 184건으로, orders의 completed 184건과 일치한다.
- 필터링 후 products 데이터와 병합했을 때, 474행 -> 474행, both 474건, left_only(미매칭) 0건이며 products.product_id 중복은 0건이다.

**3-2. 나의 해석과 판단**
- completed 주문 184건이 모두 주문 상세 정보를 가지고 있고, 모든 주문 상세 행이 상품 정보와 연결되었다.
- cancelled와 refunded는 완료되지 않거나 환불된 주문이기 때문에 completed 주문 기준 금액 계산에서 제외하였다.
- completed 주문 필터링을 먼저 하고 상품을 병합한 이유는 분석 대상인 completed 474행에 대해서만 병합 검증을 하기 위해서였다. 

**3-3. 업무·분석적 의미**
- 취소/환불 주문을 제외했기 때문에 실제로 거래가 완료된 건만 비교할 수 있다.
- 764행에서 474행으로 줄어들면서 제외된 주문이 있기 떄문에, 전체 주문 금액과 completed 금액을 구분해서 서술해야 한다.

**3-4. 한계**
- completed의 업무적인 의미를 확인하지 못했다. 결제 완료를 의미하는지, 배송 완료인지 등은 구분할 수 없다. 
- 주문 상태는 데이터를 추출한 시점의 값이기 때문에, 이후 상태가 바뀌었다면 이 시점의 결과와 다를 수 있다. 따라서 정확한 분석을 위해서는 데이터를 계속 새롭게 업데이트해야 한다.

**4. 주요 집계 수행**

In [15]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


In [16]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)

,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,14566000,16,910375.0
2,2025-11,11403000,14,814500.0
3,2025-12,20963000,22,952864.0
4,2026-01,16416000,19,864000.0
5,2026-02,3861000,6,643500.0
6,2026-03,19010000,25,760400.0
7,2026-04,10490000,16,655625.0
8,2026-05,14900000,18,827778.0
9,2026-06,14244000,17,837882.0


In [17]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


In [25]:
import pandas as pd

original_total = completed_order_sales['line_total'].sum()
totals = {
    '완료 주문 상세 line_total': original_total,
    'category_sales': category_sales['total_sales'].sum(),
    'monthly_summary': monthly_summary['total_sales'].sum(),
    'customer_sales': customer_sales['total_sales'].sum(),
}
for name, value in totals.items():
    print(f'{name:<26}{value:>15,.0f}')
print('합계 일치:', 'PASS' if all(v == original_total for v in totals.values()) else 'FAIL')


완료 주문 상세 line_total           148,990,000
category_sales                148,990,000
monthly_summary               148,990,000
customer_sales                148,990,000
합계 일치: PASS


**4-1. 결과 관찰**
- 카테고리별: total_sales 1위는 스포츠이다. 이어서 전자기기 26,400,000, 생활용품 23,915,000, 뷰티 23,383,000, 식품 16,573,000, 도서 16,389,000, 패션 10,587,000 순이다. 7개 카테고리 합계는 148,990,000이다. 금액 순위와 수량 순위는 일부 다르다. 예를 들어, 금액은 전자기기가 2위, 생활용품이 3위인데 수량은 생활용품이 2위, 전자기기가 3위이다. 
- 상품별: 1위는 스포츠 상품, 2위는 식품 상품이다. 상위 10개 상품 중 스포츠 3개, 전자기기 2개, 생활용품 2개이고, 뷰티, 식품, 패션이 각 1개이다.
- 월별: 2025-09부터 2026-09까지 13개월이며, 합계는 148,990,000으로 카테고리별 합계와 같다.total_sales 최대는 2025-12(20,963,000, 22건), 최소는 2025-09(2,650,000, 5건)이다. 주문 건수가 가장 많은 달은 2026-03(25건)이고, 월별 order_count의 합은 184건이다.
- 고객별: 1위는 Customer 117, 2위는 Customer 102이다. 상위 10명의 합계는 34,694,000이다.

**4-2. 나의 해석과 판단**
- 메인 분석 질문의 답은 스포츠이다. 2위인 전자기기와의 차이는 5,343,000으로, 전체 금액의 약 3.6%이다. 수량도 스포츠 분야가 1위인 것으로 보아, 상품 하나하나의 가격이 비싸기 때문에 금액이 1위인 것은 아니라고 판단하였다. 다만 소수 상품의 영향이 일부 있을 수도 있다. 상위 10개 표에 보이는 스포츠 상품 3개의 합이 13,205,000으로 스포츠 분야 합계의 약 42%를 차지한다.
- 4위 뷰티와 5위 식품 사이의 금액 차이는 6,810,000으로 크지만, 5위 식품과 6위 도서 사이는 184,000만큼만 차이난다. 두 카테고리의 순위는 안정적이지 않다.
- 금액 순위와 수량 순위가 다른 것은 카테고리별 단가 차이 때문이다. 전자기기는 금액은 2위, 수량은 3위이고, 생활용품은 금액 3위, 수량 2위이다. 이처럼 금액과 수량 순위가 어긋하는 카테고리들이 있다. 수량당 평균 금액을 계산해보면 식품이 가장 높다. 
- 월별 금액은 특정 달에 몰려 있다. 2025-12가 전체의 14.1%이고, 2026-03, 2026-01 순으로 많다. 주문 건수는 2026-03이 25건으로 가장 많은데, 이는 금액 순위와는 일치하지 않는다. 
- 특정 고객에 주문이 집중된 것은 아니다. 1위 고객이 전체 고객의 2.8%이고, 상위 20명을 합치면 23.3%로, 특정 고객 몇 명에게 금액이 몰리지는 않았다.
- 집계 결과가 일치한다. 카테고리별 합계와 월별 합계가 모두 148,990,000으로 같고, 월별 order_count의 합이 184건으로 orders의 completed 건수와 같다. 

**4-3. 업무·분석적 의미**
- 분석 질문에 대한 답과 근거를 제시할 수 있다. completed 주문 기준으로 금액이 가장 큰 카테고리는 스포츠이고, 수량에서도 1위이다. 카테고리별 상품 구성이나 프로모션 우선 수위를 검토할 때 이를 사용할 수 있다.
- 금액과 수량 관점을 구분해서 봐야 한다. 카테고리별 금액 순위와 수량 순위가 다른데, 매출 중심 판단과 물량 중심 판단에서 보는 카테고리가 달라진다.
- 상품별 집계는 카테고리 결과를 보완할 수 있다. 카테고리 내에서 어떤 상품이 상위권인지 보이므로, 카테고리 단위로 판단할지 개별 상품 단위로 판단할지 선택할 수 있다. 예를 들어 식품은 카테고리별 순위에서는 5위이지만, 식품 상품 012는 상품별 금액에서 전체 2위이다.
- 월별 집계는 시기별 계획에 참고할 수 있다. 금액이 큰 달과 작은 달이 뚜렷하기 때문에 재고나 인력 등을 시기별로 다르게 책정해야 하고, 프로모션 시기를 언제로 할지도 가늠할 수 있다.
- 고객별 집계에서 구매 금액이 큰 고객과 구매 건수가 많은 고객을 분리해서 관리대상을 분류한다.

**4-4. 한계**
- 이 결과는 quantity x unit_price 기준 금액이기 때문에 금액이 가장 큰 카테고리라고 헤서 이익이나 마진이 가장 큰 건 아니다. 할인/세금/배송비 등의 반영이 되지 않아, 회계상 순매출은 아니다.
- 스포츠 금액이 가장 큰 이유는 이 집계만으로 알 수 없다. 카테고리별 상품 수와 판매 기간의 차이를 고려하지 않았다.
- 상품별 해석은 싱위 10개 표만을 근거로 하였기에, 전체 상품의 분포는 확인하지 않았다.
- 월별 집계는 order_date 기준으로, 첫 달과 마지막 달이 1달을 전부 채운 기간인지 확인하지 않았다. 

**결과 저장하기**

In [18]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)

ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


In [19]:
check_category = pd.read_csv(
    REPORT_DIR / "ch04_category_sales.csv"
)

print(check_category.shape)
print(check_category.columns.tolist())

(7, 4)
['category', 'total_quantity', 'total_sales', 'sales_ratio']


**LLM 코드 검증**


LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.

In [22]:
category_sales = order_items.groupby('category')['line_total'].sum()

KeyError: 'category'

실행 불가능함. category는 products 데이터에만 있는 컬럼이므로 order_items에서 groupby할 수 없다. 
검증 순서
1. 먼저 category가 어느 테이블에 있는지 확인.
2. 병합하는 orders, products 테이블의 key 고유성 확인.
3. order_items ↔ orders 병합 (many_to_one, indicator, 행 수, 미매칭)
4. completed 필터링
5. completed ↔ products 병합 (many_to_one, indicator, 행 수, category 결측)
6. 카테고리별 집계 (dropna=False)
7. 집계 합계와 원본 completed line_total 합계 대조

아래에 안전한 수정 코드 첨부.

In [23]:
# 1) 컬럼 위치 확인
for name, df in {'order_items': order_items, 'orders': orders, 'products': products}.items():
    print(name, df.columns.tolist())

# 2) key 고유성 (병합 대상 쪽)
assert orders['order_id'].is_unique, 'orders.order_id 중복'
assert products['product_id'].is_unique, 'products.product_id 중복'

# 3) order_items ↔ orders
items = order_items.copy()
items['line_total'] = items['quantity'] * items['unit_price']

step1 = items.merge(
    orders[['order_id', 'order_status']],
    on='order_id', how='left', validate='many_to_one', indicator=True,
)
print('병합 전/후 행 수:', len(items), len(step1))
print(step1['_merge'].value_counts())
step1 = step1.drop(columns='_merge')

# 4) completed 필터
completed = step1[step1['order_status'] == 'completed'].copy()
print('completed 행 수:', len(completed))

# 5) completed ↔ products
step2 = completed.merge(
    products[['product_id', 'category']],
    on='product_id', how='left', validate='many_to_one', indicator=True,
)
print('병합 전/후 행 수:', len(completed), len(step2))
print(step2['_merge'].value_counts())
print('category 결측:', step2['category'].isna().sum())
step2 = step2.drop(columns='_merge')

# 6) 카테고리별 집계
safe_category_sales = (
    step2.groupby('category', dropna=False)['line_total']
    .sum()
    .sort_values(ascending=False)
)
display(safe_category_sales)

# 7) 합계 대조
print('원본 completed 합계:', completed['line_total'].sum())
print('카테고리별 합계   :', safe_category_sales.sum())

order_items ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
orders ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
products ['product_id', 'product_name', 'category', 'price']
병합 전/후 행 수: 764 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
completed 행 수: 474
병합 전/후 행 수: 474 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64
category 결측: 0


category
스포츠     31743000
전자기기    26400000
생활용품    23915000
뷰티      23383000
식품      16573000
도서      16389000
패션      10587000
Name: line_total, dtype: int64

원본 completed 합계: 148990000
카테고리별 합계   : 148990000


orders와 order_items를 사용해
completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.
- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기

위의 질문을 LLM에 입력했을 때 제안받은 코드를 Notebook의 실제 코드와 비교한다.

검증 항목
- LLM의 제안에서는 컬럼명이 order_status가 아니라 status였다.
- 키 관계는 맞았다. (order_id, many-to-one)
- completed 필터가 있었고, Notebook 코드와 일치했음.
- 병합 검증은 둘 다 있었다. LLM의 코드는 미매칭 금액과 items 없는 주문까지 추가로 확인.
- 집계 단위 맞았음
- 결과 의미 과장하지 않았음. 

**최종 Evidence**

[Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: [예]

2. 병합 검증
- orders.order_id 중복: [0]
- orders merge 전 행 수: [764]
- orders merge 후 행 수: [764]
- orders merge 미매칭: [0]
- products merge 전 행 수: [474]
- products merge 후 행 수: [474]
- products merge 미매칭: [0]

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: [0]

4. 교차 검증
- 완료 주문 상세 line_total 합계: [148,990,000]
- category_sales 합계: [148,990,000]
- monthly_summary 합계: [148,990,000]
- customer_sales 합계: [148,990,000]
- 합계 일치 여부: [PASS]

5. 저장 파일
- ch04_category_sales.csv: [존재]
- ch04_product_sales.csv: [존재]
- ch04_monthly_sales.csv: [존재]
- ch04_customer_sales.csv: [존재]

6. LLM 검증
- LLM이 빠뜨린 검증 항목: [LLM이 제안한 코드에서는 컬럼명이 order_status가 아니라 status였다.]

![분석 질문 및 계산 기준](images/step01_question.png)
![line_total 생성](images/step02_line_total.png)
![merge 검증](images/step03_merge.png)
![주요 groupby 결과](images/step04_groupby.png)
![총합 일치 검증](images/step05_합계일치.png)
![LLM 코드 검증](images/step06_llm.png)

7. 남은 질문
- [completed는 업무에서 어떤 상태(결제 완료, 배송 완료 등)를 뜻하는가? 이 상태는 추출 시점의 값인가? order_items.unit_price는 정가인가, 할인이 반영된 주문 시점 가격인가?스포츠의 금액이 큰 이유는 상품 수가 많아서인가, 상품당 판매 수가 많아서인가?금액 차이가 184,000으로 작은 식품과 도서의 순위는 기간이나 기준을 바꿔도 유지되는가?]

## 실습 과제 

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.

In [32]:
import pandas as pd

required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')

# 1. 40세 이상 고객 추출 (customers의 나이 컬럼명 확인: age 가정)
customers_40plus = customers[customers["age"] >= 40]
print("1) 40세 이상 고객")
print(customers_40plus.head(), "\n")

# 2. 상품 가격이 낮은 순서대로 10개
cheapest10 = products.sort_values("price", ascending=True).head(10)
print("2) 가격 낮은 상품 10개")
print(cheapest10, "\n")

# 3. 결제수단별 주문 수와 비율
pay_count = orders["payment_method"].value_counts()
pay_ratio = orders["payment_method"].value_counts(normalize=True).mul(100).round(2)
pay_summary = pd.DataFrame({"주문수": pay_count, "비율(%)": pay_ratio})
print("3) 결제수단별 주문 수/비율")
print(pay_summary, "\n")

# 4. 카테고리별 평균 상품 가격
avg_price = (
    products.groupby("category")["price"].mean().round(2).sort_values(ascending=False)
)
print("4) 카테고리별 평균 상품 가격")
print(avg_price, "\n")

# 5. 완료 주문 금액 vs 전체 주문 금액 차이
# 주문상세에 상품 가격을 붙여 행별 금액 계산
items = order_items.merge(products[["product_id", "price"]], on="product_id", how="left")
items["amount"] = items["price"] * items["quantity"]

# 주문 상태를 붙임
items = items.merge(orders[["order_id", "order_status"]], on="order_id", how="left")

total_amount = items["amount"].sum()
completed_amount = items.loc[items["order_status"] == "completed", "amount"].sum()
diff = total_amount - completed_amount

print("5) 금액 비교")
print(f"전체 주문 금액: {total_amount:,}")
print(f"완료 주문 금액: {completed_amount:,}")
print(f"차이(전체 - 완료): {diff:,}")

데이터 불러오기 완료
1) 40세 이상 고객
   customer_id name gender  age city signup_date
2            3  이경수      F   61   성남  2024-08-22
3            4  조영호      F   55   울산  2026-06-23
6            7  이상현      F   53   인천  2025-02-21
8            9  송지민      M   69   서울  2025-12-29
9           10  유도현      F   62   울산  2024-12-16 

2) 가격 낮은 상품 10개
    product_id product_name category  price
5            6  전자기기 상품 006     전자기기   5000
27          28    뷰티 상품 028       뷰티   5000
97          98   스포츠 상품 098      스포츠  10000
46          47  생활용품 상품 047     생활용품  11000
94          95  전자기기 상품 095     전자기기  20000
49          50    뷰티 상품 050       뷰티  23000
82          83  전자기기 상품 083     전자기기  24000
86          87    도서 상품 087       도서  25000
41          42    패션 상품 042       패션  28000
58          59    뷰티 상품 059       뷰티  32000 

3) 결제수단별 주문 수/비율
                주문수  비율(%)
payment_method            
kakao_pay        79  26.33
naver_pay        77  25.67
bank_transfer    74  24.67
card             70  23.3

#6 프롬프트 요청

아래 데이터 구조와 조건에 맞춰 pandas 코드를 작성해 주세요.

[데이터 구조] (아래 변수명의 DataFrame이 이미 로드되어 있으므로 파일을 다시 읽지 마세요)
- customers   (150행): customer_id, name, gender, age, city, signup_date
- products    (100행): product_id, product_name, category, price
- orders      (300행): order_id, customer_id, order_date, payment_method, order_status
- order_items (764행): order_item_id, order_id, product_id, quantity, unit_price

[참고 사항]
- orders에는 금액 컬럼이 없습니다. 금액은 order_items에서
  amount = unit_price * quantity 로 계산합니다.
- unit_price는 products.price와 전부 일치함을 이미 확인했으므로 products는 사용하지 않습니다.
- order_status의 완료 값은 "completed"입니다.

[실행 순서]
0. 병합 전 점검
1. order_items에 amount = unit_price * quantity 컬럼 추가
2. order_items와 orders를 order_id 기준으로 병합 (customer_id, order_status 붙이기)
3. order_status == "completed"인 행만 남겨 고객별 total_amount 계산
4. customers 기준 left join으로 이름(name)과 total_amount 합치기
   - 구매 이력이 없는 고객의 total_amount는 0으로 채운 뒤 int로 변환
5. total_amount 내림차순, 동점이면 customer_id 오름차순으로 정렬해
   상위 10명의 customer_id, name, total_amount 출력

[병합 검증]
- 모든 검증은 check(name, condition, value) 함수로 구현하세요.
  이 함수는 "항목명: 통과/실패 (값)" 형태로 한 줄을 출력하고 결과를 리스트에 저장합니다.
  검증 도중에는 assert로 중단하지 말고, 마지막에 실패 항목이 하나라도 있으면
  실패 항목 목록을 출력하고 종료 직전에 한 번만 assert 하세요.
- 0단계 병합 전 점검:
  · customers.customer_id, orders.order_id, order_items.order_item_id 중복 여부
  · order_items.order_id, orders.order_id, orders.customer_id, customers.customer_id 결측치 여부
  · 병합 키(order_id, customer_id)의 dtype이 양쪽 테이블에서 동일한지
- 병합 시 validate 옵션:
  · order_items ↔ orders : how="left", validate="many_to_one"
  · customers ↔ 고객별 집계 : how="left", validate="one_to_one"
- 행 수 검증 (하드코딩하지 말고 병합 전 len()과 비교):
  · 2단계 병합 후 행 수 == 병합 전 order_items 행 수 (764)
  · 4단계 병합 후 행 수 == 병합 전 customers 행 수 (150)
- 매칭 검증:
  · order_items 중 orders에 없는 order_id 건수 (0이어야 통과)
  · orders 중 customers에 없는 customer_id 건수 (0이어야 통과)
  · 2단계 병합 후 customer_id, order_status, amount의 결측치 건수 (모두 0이어야 통과)
- 최종 검증:
  · 4단계 결과의 total_amount 합계 == 3단계 직전 completed 행 전체 amount 합계

[출력 형식]
- 단계별 주석이 포함된 실행 가능한 하나의 코드 블록
- 변수명은 영어, 주석과 출력 메시지는 한국어
